In [ ]:
from datetime import datetime
from pathlib import Path
import sys

sys.path.insert(0, str(Path("../..").resolve()))

import ad_safe_lib as ad_safe

print("challenge:", ad_safe.CHALLENGE_DIR)
print("device:", ad_safe.DEVICE)
print("dataset:", ad_safe.DATA_DIR)
examples_root = ad_safe.AD_SAFE_RUNS_DIR / "notebook_examples"
examples_root.mkdir(parents=True, exist_ok=True)


# Quick From-Scratch Training

Tiny `simple_cnn` run on stratified subsets. This is for API learning and artifact shape, not model quality.


In [ ]:
run_id = "notebook-01-" + datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
output_dir = examples_root / run_id
train_source = ad_safe.DatasetSourceSpec("train", fraction=0.06, seed=1)
eval_sources = {
    "val": ad_safe.DatasetSourceSpec("val", fraction=0.10, seed=101),
    "test": ad_safe.DatasetSourceSpec("test", fraction=0.10, seed=102),
}

config = ad_safe.TrainingConfig(
    base_model="simple_cnn",
    epochs=3,
    patience=2,
    batch_size=16,
    learning_rate=(5e-4,),
    resplit_runs=1,
)
phase = ad_safe.PhaseSpec(
    job_index=0,
    phase_index=0,
    prefix="simple_cnn_quick",
    name="quick",
    requested_seed=1,
    config=config,
    unfreeze_all=True,
    model_filename="simple_cnn_quick.pt",
    history_filename="simple_cnn_quick_history.png",
    json_filename="simple_cnn_quick.json",
)
plan = ad_safe.RunPlan(
    output_dir=output_dir,
    run_id=run_id,
    train_split="train",
    eval_splits=("val", "test"),
    jobs=(
        ad_safe.JobSpec(
            job_index=0,
            job_id="simple_cnn",
            display_name="simple_cnn quick",
            backbone="simple_cnn",
            phases=(phase,),
        ),
    ),
    resume=False,
    force=True,
    setup_path=output_dir / "setup.json",
    check_foreign_contract=False,
    train_source=train_source,
    eval_sources=eval_sources,
)

result = ad_safe.run_training_plan(plan)
phase_result = result.phase_results[-1]
model_path = phase_result.model_path
history_path = phase_result.history_path
print("model:", model_path)
print("history:", history_path)
model_path


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(history_path)))
history_path


In [ ]:
eval_result = ad_safe.run_evaluation_plan(
    ad_safe.EvaluationPlan(
        models=(ad_safe.ModelEvalSpec(path=model_path, row_id="simple_cnn_quick"),),
        datasets=(
            ad_safe.DatasetEvalSpec(name="val", batch_size=8, source=eval_sources["val"]),
            ad_safe.DatasetEvalSpec(name="test", batch_size=8, source=eval_sources["test"]),
        ),
        output_dir=output_dir,
        sort_key="acc_test",
        title="Quick model metrics",
    )
)
eval_result.csv_path


In [ ]:
model = ad_safe.load_model(model_path)
test_dataset = ad_safe.load_dataset_source(eval_sources["test"])

fig = ad_safe.generate_test_figure(model, test_dataset)
ad_safe.save_figure(fig, output_dir / "sample_prediction.png")
fig


In [ ]:
adv_fig = ad_safe.generate_adversarial_example(
    model,
    test_dataset,
    strategy=ad_safe.MinimalFlipPgdStrategy(
        max_epsilon=0.18,
        num_steps=8,
        search_steps=10,
        refinement_steps=6,
    ),
    max_tries=32,
    require_correct_original=True,
)
ad_safe.save_figure(adv_fig, output_dir / "adversarial_example.png")
adv_fig


In [ ]:
reversal_fig = ad_safe.generate_class_reversal_figure(
    model,
    strategy=ad_safe.RandomRestartTargetClassStrategy(
        step_size=0.035,
        num_steps=48,
        num_restarts=4,
    ),
)
ad_safe.save_figure(reversal_fig, output_dir / "class_reversal.png")
reversal_fig
